# Tutorial 01: Building Your First Robot

[![ Click here to deploy.](https://brev-assets.s3.us-west-1.amazonaws.com/nv-lb-dark.svg)](https://brev.nvidia.com/launchable/deploy?launchableID=env-35QaaoiXx6VDmVNBOEtmpLs9VBs)

<p align="center">
  <img src="RRRP_SCARA.png" alt="RRRP SCARA Robot" width="500"/>
</p>

<p align="center"><i>
Source: <b>MODERN ROBOTICS: MECHANICS, PLANNING, AND CONTROL</b><br/>
Figure 4.12: An RRRP SCARA robot for performing pick-and-place operations.
</i></p>

---

This tutorial introduces the fundamental concepts needed to build and understand robotic systems. We'll construct an **RRRP SCARA robot** step-by-step while learning about configuration space, degrees of freedom, and joint types.

## Learning Objectives

By the end of this tutorial, you will:

1. **Understand Configuration Space** — What it means to describe a robot's state
2. **Calculate Degrees of Freedom** — Using Grübler's formula
3. **Know Joint Types** — Revolute, prismatic, spherical, and more
4. **Build a Robot Programmatically** — Using Newton's `ModelBuilder` API
5. **Export to USD** — Save the robot for visualization

This tutorial serves as a foundation for:
- **Tutorial 02**: Forward Kinematics
- **Tutorial 03**: Inverse Kinematics

---



## Setup and Imports


In [1]:
import newton
import warp as wp
import numpy as np
from tqdm.notebook import trange

# Initialize Warp
wp.init()

# Set NumPy print options for cleaner output
np.set_printoptions(precision=4, suppress=True, linewidth=100)


Warp 1.11.0.dev20251123 initialized:
   Git commit: 8b8f0b85ca54c0026574f834764e26615056aef6
   CUDA Toolkit 12.8, Driver 13.0
   Devices:
     "cpu"      : "x86_64"
     "cuda:0"   : "NVIDIA L40S" (44 GiB, sm_89, mempool enabled)
   Kernel cache:
     /root/.cache/warp/1.11.0.dev20251123


---

## Part 1: Degrees of Freedom

### Degrees of Freedom of a Rigid Body

A **rigid body** in space has:
- **6 DOF in 3D space**: 3 translations $(x, y, z)$ + 3 rotations $(\alpha, \beta, \gamma)$
- **3 DOF in 2D space**: 2 translations $(x, y)$ + 1 rotation $(\theta)$

### Degrees of Freedom of a Robot

For a robot mechanism, the DOF is determined by:
1. Number of rigid bodies (links)
2. Number and type of joints connecting them
3. Constraints imposed by the joints

### Grübler's Formula

The **Grübler formula** calculates the DOF of a mechanism:

$$\text{DOF} = m(N - 1 - J) + \sum_{i=1}^{J} f_i$$

Where:
- $m$ = DOF of a rigid body in space (6 for 3D, 3 for 2D)
- $N$ = Number of links (including the fixed ground)
- $J$ = Number of joints
- $f_i$ = DOF of joint $i$

**Alternative form for open chains (serial robots):**

$$\text{DOF} = \sum_{i=1}^{n} f_i$$

For serial robots (like robot arms), the DOF simply equals the sum of all joint DOFs.


In [2]:
def calculate_dof_grubler(n_links, joints, dim=3):
    """
    Calculate degrees of freedom using Grübler's formula.
    
    Parameters:
        n_links: Number of links INCLUDING the ground (fixed base)
        joints: List of (dof_per_joint) for each joint
        dim: Dimension (3 for spatial, 2 for planar)
    
    Returns:
        DOF of the mechanism
    """
    m = 6 if dim == 3 else 3  # DOF per rigid body
    n_joints = len(joints)
    sum_fi = sum(joints)
    
    dof = m * (n_links - 1 - n_joints) + sum_fi
    return dof


# Example 1: Simple pendulum (1 revolute joint)
# N=2 (ground + 1 link), J=1, f₁=1
dof_pendulum = calculate_dof_grubler(n_links=2, joints=[1], dim=2)
print(f"Simple Pendulum (2D): DOF = {dof_pendulum}")

# Example 2: Double pendulum (2 revolute joints)
# N=3 (ground + 2 links), J=2, f₁=f₂=1
dof_double_pendulum = calculate_dof_grubler(n_links=3, joints=[1, 1], dim=2)
print(f"Double Pendulum (2D): DOF = {dof_double_pendulum}")

# Example 3: SCARA robot (3R + 1P = 4 joints)
# N=5 (ground + 4 links), J=4, f₁=f₂=f₃=f₄=1
dof_scara = calculate_dof_grubler(n_links=5, joints=[1, 1, 1, 1], dim=3)
print(f"SCARA Robot (3D): DOF = {dof_scara}")

# For serial chains, DOF = sum of joint DOFs
print(f"\nSimplified for serial chains: DOF = Σfᵢ = {1+1+1+1} ✓")


Simple Pendulum (2D): DOF = 1
Double Pendulum (2D): DOF = 2
SCARA Robot (3D): DOF = 4

Simplified for serial chains: DOF = Σfᵢ = 4 ✓


---

## Part 2: Joint Types

Joints constrain the relative motion between two bodies. Newton supports several joint types:

### Common Joint Types

| Joint Type | DOF | Description | Newton Function |
|------------|-----|-------------|-----------------|
| **Revolute** (R) | 1 | Rotation around single axis | `add_joint_revolute()` |
| **Prismatic** (P) | 1 | Translation along single axis | `add_joint_prismatic()` |
| **Spherical** (Ball) | 3 | Rotation in all directions | `add_joint_ball()` |
| **Fixed** | 0 | Rigid connection (no motion) | `add_joint_fixed()` |
| **Free** | 6 | Floating base (all 6 DOF) | `add_joint_free()` |
| **D6** | 1-6 | Generic configurable joint | `add_joint_d6()` |

### Visual Representation

```
Revolute (Hinge):       Prismatic (Slider):     Spherical (Ball):
    ○──────○               ○════○══>              ◎──────○
    │  θ   │               │  d  │               / | \
    ↺                      ↔                    α β γ
```

### Joint Conventions in Newton

Each joint connects a **parent** body to a **child** body:
- `parent=-1` means the world/ground frame
- `parent_xform`: Transform from parent body origin to joint frame
- `child_xform`: Transform from joint frame to child body origin
- `axis`: The axis of rotation/translation (for 1-DOF joints)


In [3]:
# Discover Joint Types from Newton's API
joint_info = {
    'add_joint_revolute':  {'dof': 1, 'coords': 1, 'desc': 'Hinge joint, rotation about one axis'},
    'add_joint_prismatic': {'dof': 1, 'coords': 1, 'desc': 'Sliding joint, translation along one axis'},
    'add_joint_ball':      {'dof': 3, 'coords': 4, 'desc': 'Spherical joint, 3D rotation (quaternion)'},
    'add_joint_fixed':     {'dof': 0, 'coords': 0, 'desc': 'Rigid connection, no motion'},
    'add_joint_free':      {'dof': 6, 'coords': 7, 'desc': 'Floating base, full 6-DOF motion'},
    'add_joint_d6':        {'dof': '1-6', 'coords': '1-6', 'desc': 'Configurable joint with selectable axes'},
    'add_joint_distance':  {'dof': 1, 'coords': 1, 'desc': 'Distance constraint between bodies'},
}

print("Newton Joint Types (from newton.ModelBuilder)")
print("=" * 80)
print(f"{'Method':<22} | {'DOF':<5} | {'Coords':<6} | {'Description'}")
print("-" * 80)

for method_name, info in joint_info.items():
    if hasattr(newton.ModelBuilder, method_name):
        print(f"{method_name:<22} | {str(info['dof']):<5} | {str(info['coords']):<6} | {info['desc']}")


Newton Joint Types (from newton.ModelBuilder)
Method                 | DOF   | Coords | Description
--------------------------------------------------------------------------------
add_joint_revolute     | 1     | 1      | Hinge joint, rotation about one axis
add_joint_prismatic    | 1     | 1      | Sliding joint, translation along one axis
add_joint_ball         | 3     | 4      | Spherical joint, 3D rotation (quaternion)
add_joint_fixed        | 0     | 0      | Rigid connection, no motion
add_joint_free         | 6     | 7      | Floating base, full 6-DOF motion
add_joint_d6           | 1-6   | 1-6    | Configurable joint with selectable axes
add_joint_distance     | 1     | 1      | Distance constraint between bodies


---

## Part 3: Joint Types

Joints constrain the relative motion between two bodies. Newton supports several joint types:

### Common Joint Types

| Joint Type | DOF | Description | Newton Function |
|------------|-----|-------------|-----------------|
| **Revolute** (R) | 1 | Rotation around single axis | `add_joint_revolute()` |
| **Prismatic** (P) | 1 | Translation along single axis | `add_joint_prismatic()` |
| **Spherical** (Ball) | 3 | Rotation in all directions | `add_joint_ball()` |
| **Fixed** | 0 | Rigid connection (no motion) | `add_joint_fixed()` |
| **Free** | 6 | Floating base (all 6 DOF) | `add_joint_free()` |
| **D6** | 1-6 | Generic configurable joint | `add_joint_d6()` |

### Visual Representation

```
Revolute (Hinge):       Prismatic (Slider):     Spherical (Ball):
    ○──────○               ○════○══>              ◎──────○
    │  θ   │               │  d  │               / | \
    ↺                      ↔                    α β γ
```

### Joint Conventions in Newton

Each joint connects a **parent** body to a **child** body:
- `parent=-1` means the world/ground frame
- `parent_xform`: Transform from parent body origin to joint frame
- `child_xform`: Transform from joint frame to child body origin
- `axis`: The axis of rotation/translation (for 1-DOF joints)


In [4]:
# ============================================================
# Discover Joint Types from Newton's API
# ============================================================

# Joint info extracted from Newton's docstrings
joint_info = {
    'add_joint_revolute':  {'dof': 1, 'coords': 1, 'desc': 'Hinge joint, rotation about one axis'},
    'add_joint_prismatic': {'dof': 1, 'coords': 1, 'desc': 'Sliding joint, translation along one axis'},
    'add_joint_ball':      {'dof': 3, 'coords': 4, 'desc': 'Spherical joint, 3D rotation (quaternion)'},
    'add_joint_fixed':     {'dof': 0, 'coords': 0, 'desc': 'Rigid connection, no motion'},
    'add_joint_free':      {'dof': 6, 'coords': 7, 'desc': 'Floating base, full 6-DOF motion'},
    'add_joint_d6':        {'dof': '1-6', 'coords': '1-6', 'desc': 'Configurable joint with selectable axes'},
    'add_joint_distance':  {'dof': 1, 'coords': 1, 'desc': 'Distance constraint between bodies'},
}

# Verify these methods exist in Newton
print("Newton Joint Types (from newton.ModelBuilder)")
print("=" * 80)
print(f"{'Method':<22} | {'DOF':<5} | {'Coords':<6} | {'Description'}")
print("-" * 80)

for method_name, info in joint_info.items():
    if hasattr(newton.ModelBuilder, method_name):
        exists = "✓"
        print(f"{method_name:<22} | {str(info['dof']):<5} | {str(info['coords']):<6} | {info['desc']}")

# Show any additional joint methods we might have missed
all_joint_methods = [m for m in dir(newton.ModelBuilder) if m.startswith('add_joint')]
extra = set(all_joint_methods) - set(joint_info.keys())
if extra:
    print(f"\nOther joint methods: {', '.join(extra)}")


Newton Joint Types (from newton.ModelBuilder)
Method                 | DOF   | Coords | Description
--------------------------------------------------------------------------------
add_joint_revolute     | 1     | 1      | Hinge joint, rotation about one axis
add_joint_prismatic    | 1     | 1      | Sliding joint, translation along one axis
add_joint_ball         | 3     | 4      | Spherical joint, 3D rotation (quaternion)
add_joint_fixed        | 0     | 0      | Rigid connection, no motion
add_joint_free         | 6     | 7      | Floating base, full 6-DOF motion
add_joint_d6           | 1-6   | 1-6    | Configurable joint with selectable axes
add_joint_distance     | 1     | 1      | Distance constraint between bodies

Other joint methods: add_joint


---

## Part 4: Building a SCARA Robot

Now let's build a **SCARA (Selective Compliance Assembly Robot Arm)** robot from scratch.

SCARA robots are commonly used for:
- Pick-and-place operations
- Assembly tasks
- Semiconductor manufacturing

### SCARA Configuration: RRRP

| Joint | Type | Description |
|-------|------|-------------|
| θ₁ | Revolute (R) | Shoulder rotation around vertical axis |
| θ₂ | Revolute (R) | Elbow rotation around vertical axis |
| θ₃ | Revolute (R) | Wrist rotation around vertical axis |
| d₄ | Prismatic (P) | Vertical extension of the end-effector |

All revolute axes are **vertical (Z-axis)**, giving the SCARA its characteristic motion.

```
      Top View:                    Side View:
           θ₂                           
      ○───────○   L₂                    ║ L₀ (base column)
     / θ₁                               ║
    ○───────────○                  ═════╬═════ L₁
    │   L₁     θ₃                       │ d₄ (prismatic)
    │                                   │
   base                              gripper (ee)
```

### Structure

```
[World] → Base → L0 (column) → L1 [θ₁] → L2 [θ₂] → Sleeve [θ₃] → Piston [d₄] → EE
```


In [5]:
# SCARA Robot Parameters
L0_HEIGHT = 0.43     # Base column height
L1_LENGTH = 0.30     # First arm length
L2_LENGTH = 0.20     # Second arm length
LINK_RADIUS = 0.05   # Link radius
LINK_HEIGHT = 0.02   # Link half-height

# Joint limits
JOINT_LIMITS = {
    "shoulder": (-np.pi/2, np.pi/2),     # θ₁: [-90°, 90°]
    "elbow": (-np.pi/4, np.pi/4),        # θ₂: [-45°, 45°]
    "wrist": (-2*np.pi/3, np.pi),        # θ₃: [-120°, 180°]
    "prismatic": (0.0, 0.25),            # d₄: [0, 25cm]
}

print("RRRP SCARA Robot Parameters")
print("=" * 50)
print(f"  L₀ (base height):     {L0_HEIGHT:.3f} m")
print(f"  L₁ (first arm):       {L1_LENGTH:.3f} m")
print(f"  L₂ (second arm):      {L2_LENGTH:.3f} m")
print(f"\nTotal reach: {L1_LENGTH + L2_LENGTH:.3f} m")
print(f"DOF: 4 (3 revolute + 1 prismatic)")
print(f"\nJoint limits:")
print(f"  θ₁ (shoulder): [{np.degrees(JOINT_LIMITS['shoulder'][0]):.0f}°, {np.degrees(JOINT_LIMITS['shoulder'][1]):.0f}°]")
print(f"  θ₂ (elbow):    [{np.degrees(JOINT_LIMITS['elbow'][0]):.0f}°, {np.degrees(JOINT_LIMITS['elbow'][1]):.0f}°]")
print(f"  θ₃ (wrist):    [{np.degrees(JOINT_LIMITS['wrist'][0]):.0f}°, {np.degrees(JOINT_LIMITS['wrist'][1]):.0f}°]")
print(f"  θ₄ (prismatic): [{JOINT_LIMITS['prismatic'][0]*100:.0f}cm, {JOINT_LIMITS['prismatic'][1]*100:.0f}cm]")


RRRP SCARA Robot Parameters
  L₀ (base height):     0.430 m
  L₁ (first arm):       0.300 m
  L₂ (second arm):      0.200 m

Total reach: 0.500 m
DOF: 4 (3 revolute + 1 prismatic)

Joint limits:
  θ₁ (shoulder): [-90°, 90°]
  θ₂ (elbow):    [-45°, 45°]
  θ₃ (wrist):    [-120°, 180°]
  θ₄ (prismatic): [0cm, 25cm]


### Building the Robot with ModelBuilder

Newton's `ModelBuilder` provides a fluent API for constructing robots:

1. `add_articulation()` — Start a new kinematic chain
2. `add_body()` — Add a rigid body (link)
3. `add_shape_*()` — Add collision/visual geometry to a body
4. `add_joint_*()` — Connect bodies with joints
5. `finalize()` — Build the simulation-ready model


In [6]:
# ============================================================
# Build SCARA Robot Using Newton's ModelBuilder
# (Matching scara_rrrp.xml - without table)
# ============================================================

# Create a fresh builder each time this cell runs
builder = newton.ModelBuilder()
builder.add_articulation(key="scara_robot")

# Dimensions from MJCF (scara_rrrp.xml)
# Note: No table in programmatic version (starts at ground level)
BASE_PLATE_HEIGHT = 0.02   # Base plate thickness
L0_HEIGHT = 0.43           # L0 cylinder height (0.215 * 2)
PISTON_HALF_HEIGHT = 0.15
EE_OFFSET = 0.30
FINGER_LENGTH = 0.05
FINGER_WIDTH = 0.02
FINGER_OFFSET = 0.045      # LINK_HEIGHT + FINGER_LENGTH/2

# ----------------------
# Base Plate (Fixed to World)
# MJCF: box size="0.150 0.150 0.020" at pos="0 0 0.520"
# Without table, we place it at ground level
# ----------------------
base = builder.add_body(key="base")
builder.add_shape_box(
    body=base,
    hx=0.15, hy=0.15, hz=BASE_PLATE_HEIGHT,
    xform=wp.transform(wp.vec3(0.0, 0.0, BASE_PLATE_HEIGHT), wp.quat_identity()),
)
builder.add_joint_fixed(parent=-1, child=base, key="base_fixed")

# ----------------------
# L0: Vertical Column (on top of base)
# MJCF: cylinder size="0.050 0.215" at pos="0 0 0.215"
# ----------------------
L0 = builder.add_body(key="L0")
builder.add_shape_cylinder(
    body=L0,
    radius=LINK_RADIUS,
    half_height=L0_HEIGHT / 2,
    xform=wp.transform(wp.vec3(0.0, 0.0, L0_HEIGHT / 2), wp.quat_identity()),
)
# L0 is fixed to base (no joint motion)
builder.add_joint_fixed(
    parent=base, child=L0,
    parent_xform=wp.transform(wp.vec3(0.0, 0.0, 2*BASE_PLATE_HEIGHT), wp.quat_identity()),
    key="L0_fixed"
)

# ----------------------
# Link 1: L1 (First Arm)
# MJCF: box size="0.150 0.050 0.020" -> hx=0.15, hy=0.05, hz=0.02
# Plus cylinder joints at both ends
# ----------------------
L1 = builder.add_body(key="L1")
# Main arm box (MJCF: size="0.150 0.050 0.020" - half-extents)
builder.add_shape_box(
    body=L1,
    hx=L1_LENGTH / 2,  # 0.15
    hy=LINK_RADIUS,    # 0.05
    hz=LINK_HEIGHT,    # 0.02 (same as base plate)
    xform=wp.transform(wp.vec3(L1_LENGTH / 2, 0.0, 0.0), wp.quat_identity()),
)
# Cylinder at start (joint location)
builder.add_shape_cylinder(
    body=L1,
    radius=LINK_RADIUS,
    half_height=LINK_HEIGHT,
    xform=wp.transform(wp.vec3(0.0, 0.0, 0.0), wp.quat_identity()),
)
# Cylinder at end
builder.add_shape_cylinder(
    body=L1,
    radius=LINK_RADIUS,
    half_height=LINK_HEIGHT,
    xform=wp.transform(wp.vec3(L1_LENGTH, 0.0, 0.0), wp.quat_identity()),
)

builder.add_joint_revolute(
    parent=L0, child=L1,
    axis=wp.vec3(0.0, 0.0, 1.0),
    parent_xform=wp.transform(wp.vec3(0.0, 0.0, L0_HEIGHT), wp.quat_identity()),
    child_xform=wp.transform_identity(),
    limit_lower=JOINT_LIMITS["shoulder"][0],
    limit_upper=JOINT_LIMITS["shoulder"][1],
    key="joint1_shoulder"
)

# ----------------------
# Link 2: L2 (Second Arm)
# MJCF: box size="0.100 0.050 0.020" -> hx=0.10, hy=0.05, hz=0.02
# Plus cylinder joints at both ends
# ----------------------
L2 = builder.add_body(key="L2")
# Main arm box (MJCF: size="0.100 0.050 0.020" - half-extents)
builder.add_shape_box(
    body=L2,
    hx=L2_LENGTH / 2,  # 0.10
    hy=LINK_RADIUS,    # 0.05
    hz=LINK_HEIGHT,    # 0.02 (same as base plate)
    xform=wp.transform(wp.vec3(L2_LENGTH / 2, 0.0, 0.0), wp.quat_identity()),
)
# Cylinder at start
builder.add_shape_cylinder(
    body=L2,
    radius=LINK_RADIUS,
    half_height=LINK_HEIGHT,
    xform=wp.transform(wp.vec3(0.0, 0.0, 0.0), wp.quat_identity()),
)
# Cylinder at end
builder.add_shape_cylinder(
    body=L2,
    radius=LINK_RADIUS,
    half_height=LINK_HEIGHT,
    xform=wp.transform(wp.vec3(L2_LENGTH, 0.0, 0.0), wp.quat_identity()),
)

builder.add_joint_revolute(
    parent=L1, child=L2,
    axis=wp.vec3(0.0, 0.0, 1.0),
    parent_xform=wp.transform(wp.vec3(L1_LENGTH, 0.0, 2*LINK_HEIGHT), wp.quat_identity()),
    child_xform=wp.transform_identity(),
    limit_lower=JOINT_LIMITS["elbow"][0],
    limit_upper=JOINT_LIMITS["elbow"][1],
    key="joint2_elbow"
)

# ----------------------
# Link 3: Sleeve (Wrist)
# ----------------------
sleeve = builder.add_body(key="sleeve")
builder.add_shape_cylinder(
    body=sleeve,
    radius=LINK_RADIUS / 2,
    half_height=LINK_HEIGHT,
)
builder.add_joint_revolute(
    parent=L2, child=sleeve,
    axis=wp.vec3(0.0, 0.0, 1.0),
    parent_xform=wp.transform(wp.vec3(L2_LENGTH, 0.0, -2*LINK_HEIGHT), wp.quat_identity()),
    child_xform=wp.transform_identity(),
    limit_lower=JOINT_LIMITS["wrist"][0],
    limit_upper=JOINT_LIMITS["wrist"][1],
    key="joint3_wrist"
)

# ----------------------
# Link 4: Piston (Prismatic)
# ----------------------
piston = builder.add_body(key="piston")
builder.add_shape_cylinder(
    body=piston,
    radius=LINK_RADIUS / 4,  # 0.025
    half_height=PISTON_HALF_HEIGHT,
    xform=wp.transform(wp.vec3(0.0, 0.0, -PISTON_HALF_HEIGHT), wp.quat_identity()),
)
builder.add_joint_prismatic(
    parent=sleeve, child=piston,
    axis=wp.vec3(0.0, 0.0, 1.0),
    limit_lower=JOINT_LIMITS["prismatic"][0],
    limit_upper=JOINT_LIMITS["prismatic"][1],
    key="joint4_prismatic"
)

# ----------------------
# Link 5: End-Effector with Palm + Fingers
# MJCF: palm size="0.075 0.050 0.020", fingers at ±0.065
# ----------------------
ee = builder.add_body(key="ee")

# Palm (gold)
builder.add_shape_box(
    body=ee,
    hx=0.075, hy=0.05, hz=LINK_HEIGHT,
    xform=wp.transform_identity(),
)

# Left finger (at +X edge, extends down)
builder.add_shape_box(
    body=ee,
    hx=0.01, hy=0.05, hz=0.025,
    xform=wp.transform(wp.vec3(0.065, 0.0, -FINGER_OFFSET), wp.quat_identity()),
)

# Right finger (at -X edge, extends down)
builder.add_shape_box(
    body=ee,
    hx=0.01, hy=0.05, hz=0.025,
    xform=wp.transform(wp.vec3(-0.065, 0.0, -FINGER_OFFSET), wp.quat_identity()),
)

# EE Site (tip of fingers for FK reference) - using a small sphere
builder.add_shape_sphere(
    body=ee,
    radius=0.003,
    xform=wp.transform(wp.vec3(0.0, 0.0, -0.070), wp.quat_identity()),
)

builder.add_joint_fixed(
    parent=piston, child=ee,
    parent_xform=wp.transform(wp.vec3(0.0, 0.0, -EE_OFFSET), wp.quat_identity()),
    child_xform=wp.transform_identity(),
    key="ee_fixed"
)

# ----------------------
# Set Initial Configuration
# ----------------------
builder.joint_q[0] = 0.0   # θ₁ = 0°
builder.joint_q[1] = 0.0   # θ₂ = 0°
builder.joint_q[2] = 0.0   # θ₃ = 0°
builder.joint_q[3] = 0.1   # θ₄ = 10cm

# ----------------------
# Finalize Model
# ----------------------
model = builder.finalize()
state = model.state()
newton.eval_fk(model, model.joint_q, model.joint_qd, state)

print("=" * 50)
print("RRRP SCARA Robot Built Successfully!")
print("=" * 50)
print(f"  Bodies: {model.body_count}")
print(f"  Joints: {model.joint_count}")
print(f"  DOF: {model.joint_dof_count}")
print(f"  Shapes: {model.shape_count}")
print(f"\nBody Names:")
for i, key in enumerate(model.body_key):
    print(f"  [{i}] {key}")
print(f"\nJoint Names:")
for i, key in enumerate(model.joint_key):
    print(f"  [{i}] {key}")


Module validate_and_correct_inertia_kernel_4e499976 4e49997 load on device 'cuda:0' took 549.45 ms  (compiled)
Module count_contact_points_cf171c27 cf171c2 load on device 'cuda:0' took 157.70 ms  (compiled)
Module newton._src.sim.articulation 46448df load on device 'cuda:0' took 6759.29 ms  (compiled)
RRRP SCARA Robot Built Successfully!
  Bodies: 7
  Joints: 7
  DOF: 4
  Shapes: 14

Body Names:
  [0] base
  [1] L0
  [2] L1
  [3] L2
  [4] sleeve
  [5] piston
  [6] ee

Joint Names:
  [0] base_fixed
  [1] L0_fixed
  [2] joint1_shoulder
  [3] joint2_elbow
  [4] joint3_wrist
  [5] joint4_prismatic
  [6] ee_fixed


### Visualizing the Robot

Let's visualize our SCARA robot using Newton's Rerun viewer.


In [7]:
# Create the viewer
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer.set_model(model)
viewer.log_state(state)

print("Robot visualized in Rerun viewer")
viewer


Module newton._src.viewer.kernels 1205f75 load on device 'cuda:0' took 1236.41 ms  (compiled)
Robot visualized in Rerun viewer


HTML(value='<div id="6ef96de4-dc52-43fd-a184-f8a89b663106"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

---

## Part 5: Exploring Configuration Space

Let's animate the robot through its configuration space to see how joint values map to end-effector positions.


In [8]:
# ============================================================
# Animate Through Configuration Space
# ============================================================

viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer.set_model(model)

# Animation parameters
n_frames = 200
sim_time = 0.0
frame_dt = 1.0 / 30.0

print("Animating robot through configuration space...")

for i in trange(n_frames, desc="Animating"):
    # Vary joint angles sinusoidally
    t = i / n_frames * 2 * np.pi
    
    q = np.array([
        np.radians(45) * np.sin(t),           # Shoulder: ±45°
        np.radians(30) * np.sin(1.5 * t),     # Elbow: ±30° (faster)
        np.radians(60) * np.sin(0.7 * t),     # Wrist: ±60° (slower)
        0.075 + 0.05 * np.sin(2 * t),         # Prismatic: 2.5-12.5 cm
    ])
    
    # Update joint configuration
    state.joint_q.assign(q)
    newton.eval_fk(model, state.joint_q, state.joint_qd, state)
    
    # Log to viewer
    viewer.begin_frame(sim_time)
    viewer.log_state(state)
    viewer.end_frame()
    
    sim_time += frame_dt

print(f"\n✓ Animation complete: {n_frames} frames")
viewer


Animating robot through configuration space...


Animating:   0%|          | 0/200 [00:00<?, ?it/s]


✓ Animation complete: 200 frames


HTML(value='<div id="a4f4dc04-92a2-4b18-9936-be29ef9ba796"><style onload="eval(atob(\'KGFzeW5jIGZ1bmN0aW9uICgp…

---

## Part 6: Task Space and Workspace

### Task Space vs Configuration Space

| | Configuration Space | Task Space |
|---|---|---|
| **Definition** | All possible joint configurations | Space where task is defined |
| **Coordinates** | Joint angles $(\theta_1, \theta_2, \ldots)$ | Position/orientation $(x, y, z, \phi, \theta, \psi)$ |
| **Dimension** | Number of DOF | Usually ≤ 6 |
| **Example** | SCARA: 4D $(\theta_1, \theta_2, \theta_3, d_4)$ | Cartesian: $(x, y, z)$ |

### Workspace

The **workspace** is the set of all positions reachable by the end-effector:

- **Reachable workspace**: All positions the EE can reach (ignoring orientation)
- **Dexterous workspace**: Positions reachable with any orientation

For our SCARA robot:
- **Horizontal**: Annular region with $|L_1 - L_2| \leq r \leq L_1 + L_2$
- **Vertical**: Range determined by prismatic joint stroke


In [9]:
# ============================================================
# Calculate and Visualize Workspace (matching Tutorial 08)
# ============================================================

# Workspace parameters (same as Tutorial 08)
r_min = abs(L1_LENGTH - L2_LENGTH)  # Inner radius: 0.10 m
r_max = L1_LENGTH + L2_LENGTH       # Outer radius: 0.50 m
z_min = L0_HEIGHT + JOINT_LIMITS["prismatic"][0]  # Min height
z_max = L0_HEIGHT + JOINT_LIMITS["prismatic"][1]  # Max height

print("SCARA Workspace Analysis")
print("=" * 50)
print(f"\nHorizontal (X-Y plane):")
print(f"  Inner radius: {r_min:.2f} m")
print(f"  Outer radius: {r_max:.2f} m")
print(f"  Shape: Annular ring")

print(f"\nVertical (Z-axis):")
print(f"  Min height: {z_min:.2f} m")
print(f"  Max height: {z_max:.2f} m")
print(f"  Stroke: {JOINT_LIMITS['prismatic'][1] - JOINT_LIMITS['prismatic'][0]:.2f} m")

# Calculate workspace volume (approximate as cylindrical shell)
workspace_area = np.pi * (r_max**2 - r_min**2)  # Horizontal area
workspace_volume = workspace_area * (z_max - z_min)  # Volume

print(f"\nWorkspace Volume:")
print(f"  Horizontal area: {workspace_area*1e4:.1f} cm²")
print(f"  Approximate volume: {workspace_volume*1e6:.1f} cm³")


SCARA Workspace Analysis

Horizontal (X-Y plane):
  Inner radius: 0.10 m
  Outer radius: 0.50 m
  Shape: Annular ring

Vertical (Z-axis):
  Min height: 0.43 m
  Max height: 0.68 m
  Stroke: 0.25 m

Workspace Volume:
  Horizontal area: 7539.8 cm²
  Approximate volume: 188495.6 cm³


In [ ]:
import matplotlib.pyplot as plt

# ============================================================
# Visualize Workspace (Top View)
# ============================================================

fig, axes = plt.subplots(1, 2, figsize=(14, 6))

# --- Left: Top View (X-Y Workspace) ---
ax1 = axes[0]

# Draw workspace annulus
theta_circle = np.linspace(0, 2*np.pi, 100)
x_outer = r_max * np.cos(theta_circle)
y_outer = r_max * np.sin(theta_circle)
x_inner = r_min * np.cos(theta_circle)
y_inner = r_min * np.sin(theta_circle)

ax1.fill(x_outer, y_outer, alpha=0.3, color='blue', label='Reachable workspace')
ax1.fill(x_inner, y_inner, alpha=1.0, color='white')  # Inner hole
ax1.plot(x_outer, y_outer, 'b-', linewidth=2)
ax1.plot(x_inner, y_inner, 'b--', linewidth=1)

# Draw robot arm at current config
theta1 = initial_q["shoulder"]
theta2 = initial_q["elbow"]

# Joint positions
p0 = np.array([0, 0])
p1 = p0 + L1_LENGTH * np.array([np.cos(theta1), np.sin(theta1)])
p2 = p1 + L2_LENGTH * np.array([np.cos(theta1 + theta2), np.sin(theta1 + theta2)])

ax1.plot([p0[0], p1[0]], [p0[1], p1[1]], 'r-', linewidth=4, label='Arm 1')
ax1.plot([p1[0], p2[0]], [p1[1], p2[1]], 'g-', linewidth=4, label='Arm 2')
ax1.scatter(*p0, s=150, c='black', zorder=5, label='Base')
ax1.scatter(*p1, s=100, c='red', zorder=5)
ax1.scatter(*p2, s=120, c='orange', zorder=5, label='End-effector')

ax1.set_xlabel('X (m)')
ax1.set_ylabel('Y (m)')
ax1.set_title('SCARA Workspace (Top View)', fontweight='bold')
ax1.set_aspect('equal')
ax1.legend(loc='upper left')
ax1.grid(True, alpha=0.3)
ax1.set_xlim(-0.6, 0.6)
ax1.set_ylim(-0.6, 0.6)

# --- Right: Side View (X-Z Workspace) ---
ax2 = axes[1]

# Draw workspace rectangle (side view)
workspace_rect = plt.Rectangle(
    (r_min, z_min), r_max - r_min, z_max - z_min,
    alpha=0.3, color='blue', label='Reachable workspace'
)
ax2.add_patch(workspace_rect)

# Draw base column
ax2.plot([0, 0], [0, L0], 'k-', linewidth=6, label='Base')

# Draw arm at current position
r_current = np.sqrt(p2[0]**2 + p2[1]**2)
z_current = z_max - initial_q["prismatic"]
ax2.plot([0, r_current], [L0, L0], 'r-', linewidth=3, label='Arms')
ax2.plot([r_current, r_current], [L0, z_current], 'm-', linewidth=2, label='Prismatic')
ax2.scatter(r_current, z_current, s=120, c='orange', zorder=5, label='End-effector')

ax2.set_xlabel('Radial distance (m)')
ax2.set_ylabel('Z (m)')
ax2.set_title('SCARA Workspace (Side View)', fontweight='bold')
ax2.legend(loc='upper right')
ax2.grid(True, alpha=0.3)
ax2.set_xlim(-0.1, 0.6)
ax2.set_ylim(-0.1, 0.5)

plt.tight_layout()
plt.savefig('scara_workspace.png', dpi=150, bbox_inches='tight')
plt.show()

print("\n✓ Workspace visualization saved as 'scara_workspace.png'")


---

## Summary

You've built a complete SCARA robot from scratch using Newton's `ModelBuilder` API!

### What You Learned
- **Configuration Space**: How robot configurations are represented
- **Degrees of Freedom**: Using Grübler's formula to count DOF
- **Joint Types**: Revolute, prismatic, and other joints in Newton
- **Robot Building**: Creating bodies, shapes, and joints programmatically
- **Workspace**: Reachable area of the end-effector

### Next Steps
- **[Tutorial 02: Forward Kinematics](02_forward_kinematics.ipynb)** - Compute end-effector position from joint angles
- **[Tutorial 03: Inverse Kinematics](03_inverse_kinematics.ipynb)** - Find joint angles for desired positions


In [ ]:
# Placeholder - cell to be deleted


# Reset state to home position
state.joint_q.assign([0.0, 0.0, 0.0, 0.1])
newton.eval_fk(model, state.joint_q, state.joint_qd, state)

# Display robot in Rerun viewer
viewer = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer.set_model(model)
viewer.log_state(state)
print("SCARA robot built successfully!")
viewer


In [13]:
# Export robot to USD file
try:
    usd_viewer = newton.viewer.ViewerUSD(output_path="RRRP_SCARA.usd", fps=30, up_axis="Z")
    usd_viewer.set_model(model)
    
    # Log initial state
    usd_viewer.begin_frame(0.0)
    usd_viewer.log_state(state)
    usd_viewer.end_frame()
    
    # Log a short animation
    for i in range(30):
        t = i / 30 * 2 * np.pi
        q = np.array([
            np.radians(30) * np.sin(t),
            np.radians(20) * np.sin(1.5 * t),
            np.radians(40) * np.sin(0.7 * t),
            0.1 + 0.05 * np.sin(2 * t),
        ])
        state.joint_q.assign(q)
        newton.eval_fk(model, state.joint_q, state.joint_qd, state)
        
        usd_viewer.begin_frame((i + 1) / 30.0)
        usd_viewer.log_state(state)
        usd_viewer.end_frame()
    
    usd_viewer.close()
    print("✓ Robot exported to RRRP_SCARA.usd")
    
except ImportError:
    print("Note: USD export requires 'usd-core' package")
    print("Install with: pip install usd-core")


USD output saved in: /workspaces/newton/tutorial/RRRP_SCARA.usd
✓ Robot exported to RRRP_SCARA.usd


In [ ]:
# Reset state to home position and display final robot
state.joint_q.assign([0.0, 0.0, 0.0, 0.1])
newton.eval_fk(model, state.joint_q, state.joint_qd, state)

# Display in Rerun viewer
viewer_final = newton.viewer.ViewerRerun(keep_historical_data=True)
viewer_final.set_model(model)
viewer_final.log_state(state)

print("Final robot visualization")
viewer_final


---

## Summary

### Key Concepts Covered

| Concept | Description |
|---------|-------------|
| **Configuration Space** | Set of all possible robot configurations $q = (\theta_1, \ldots, \theta_n)$ |
| **Degrees of Freedom** | Number of independent parameters needed to describe configuration |
| **Grübler's Formula** | $\text{DOF} = m(N-1-J) + \sum f_i$ |
| **Joint Types** | Revolute (1 DOF), Prismatic (1 DOF), Spherical (3 DOF), etc. |
| **Workspace** | Set of all reachable end-effector positions |

### Newton API Summary

```python
# Create robot
builder = newton.ModelBuilder()
builder.add_articulation(key="my_robot")

# Add links and joints
link = builder.add_body(key="link_name")
builder.add_shape_box(link, hx=..., hy=..., hz=...)
builder.add_joint_revolute(parent, child, axis=wp.vec3(0,0,1), ...)

# Finalize and simulate
model = builder.finalize()
state = model.state()
newton.eval_fk(model, state.joint_q, state.joint_qd, state)
```

### Next Steps

- **[Tutorial 02: Forward Kinematics](02_forward_kinematics.ipynb)**: Compute end-effector position from joint angles using DH parameters
- **[Tutorial 03: Inverse Kinematics](03_inverse_kinematics.ipynb)**: Find joint angles for a desired end-effector pose

### References

- Lynch, K.M. and Park, F.C. (2017). *Modern Robotics: Mechanics, Planning, and Control*. Cambridge University Press. [http://modernrobotics.org](http://modernrobotics.org)


In [ ]:
print("✓ Tutorial 01: Building Your First Robot - Complete!")
print("\n  Next: Tutorial 02 (Forward Kinematics)")
